# Course 6 — Support Vector Machines

Maximum-margin classifiers, the kernel trick, and SVM in the p ≫ n
regime on the Khan gene-expression dataset.

**Sections**
1. Maximum-margin classifier and support vectors (0:00–0:30)
2. Soft margin and kernels (0:30–1:00)
3. SVM on Khan — p ≫ n (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import accuracy_score, classification_report


## 1. Maximum-margin classifier

With two linearly separable classes, there are infinitely many
separating hyperplanes. The maximum-margin one is the unique line that
is as far as possible from the nearest point of each class. The points
that touch the margin are the **support vectors**.

In [ ]:
X, y = make_blobs(n_samples=80, centers=2, cluster_std=0.8, random_state=0)
svc = SVC(kernel='linear', C=100).fit(X, y)

# Plot data, boundary, margins, and support vectors
xx = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200)
w = svc.coef_[0]; b = svc.intercept_[0]
yy = -(w[0] * xx + b) / w[1]
margin = 1 / np.linalg.norm(w)
yy_up = yy + margin * np.sqrt(1 + (w[0]/w[1])**2)
yy_dn = yy - margin * np.sqrt(1 + (w[0]/w[1])**2)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[y==0,0], X[y==0,1], s=30, label='class 0')
ax.scatter(X[y==1,0], X[y==1,1], s=30, color='C3', label='class 1')
ax.plot(xx, yy, 'k-'); ax.plot(xx, yy_up, 'k--', alpha=0.5)
ax.plot(xx, yy_dn, 'k--', alpha=0.5)
ax.scatter(svc.support_vectors_[:,0], svc.support_vectors_[:,1],
            s=120, facecolors='none', edgecolors='black', label='SV')
ax.legend(); ax.set_title('Maximum-margin classifier'); plt.show()
print(f'#support vectors: {svc.n_support_}  ->  total {len(svc.support_vectors_)}')


## 2. Soft margin and kernels

**Soft margin (`C`)** — small C lets points violate the margin in
exchange for a wider gap. Large C is hard about the margin (few SVs,
low bias, high variance).

In [ ]:
# Two C values on the moons dataset to show the C effect
Xm, ym = make_moons(n_samples=200, noise=0.3, random_state=0)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, kernel in zip(axes, ['linear', 'poly', 'rbf']):
    svc = Pipeline([('s', StandardScaler()),
                    ('svc', SVC(kernel=kernel, degree=3, C=1, gamma='scale'))]).fit(Xm, ym)
    xx, yy = np.meshgrid(np.linspace(Xm[:,0].min()-0.3, Xm[:,0].max()+0.3, 200),
                          np.linspace(Xm[:,1].min()-0.3, Xm[:,1].max()+0.3, 200))
    Z = svc.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25)
    ax.scatter(Xm[ym==0,0], Xm[ym==0,1], s=12)
    ax.scatter(Xm[ym==1,0], Xm[ym==1,1], s=12, color='C3')
    ax.set_title(f'{kernel} kernel — acc={svc.score(Xm, ym):.3f}')
plt.tight_layout(); plt.show()


Linear can't bend; polynomial bends but rigidly; RBF curves to fit the
moons almost exactly. The kernel is the heart of SVM.

## 3. SVM on Khan — p ≫ n

Khan's gene-expression study: 63 training cases × 2308 features, 4
cancer subtypes. Classic p ≫ n: more predictors than samples. SVM
thrives here; an unregularized logistic regression would diverge.

In [ ]:
xtr = load_dataset('khan-xtrain').to_numpy()
ytr = load_dataset('khan-ytrain')['x'].to_numpy()
xte = load_dataset('khan-xtest').to_numpy()
yte = load_dataset('khan-ytest')['x'].to_numpy()
print('train:', xtr.shape, '  test:', xte.shape)
print('classes:', np.bincount(ytr)[1:])  # subtypes labelled 1..4


In [ ]:
svc = SVC(kernel='linear', C=10).fit(xtr, ytr)
print(f'train accuracy = {svc.score(xtr, ytr):.4f}')
print(f'test accuracy  = {svc.score(xte, yte):.4f}')
print('confusion matrix on test:')
from sklearn.metrics import confusion_matrix
print(confusion_matrix(yte, svc.predict(xte)))


### Tune C with a small grid (cv=3 because n=63)

In [ ]:
grid = GridSearchCV(SVC(kernel='linear'), {'C': [0.1, 1, 10, 100]},
                     cv=3, n_jobs=-1).fit(xtr, ytr)
print(f'best C = {grid.best_params_["C"]}  CV acc = {grid.best_score_:.4f}')
print(f'test acc with best C = {grid.best_estimator_.score(xte, yte):.4f}')


### Recap
- The maximum-margin classifier is unique; support vectors define it.
- `C` softens the margin; small C = wide margin, large C = strict.
- The kernel trick lets a linear method fit non-linear boundaries.
- SVMs are *the* tool when p ≫ n. Use a linear kernel and tune C.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Build `Pipeline(StandardScaler → SVC(kernel='rbf'))` and
fit it on `penguins` (drop NaN) to predict `species` from the four
numeric measurements. Report 5-fold CV accuracy.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
df = load_dataset('penguins').dropna()
X = df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']]
y = df['species']
pipe = Pipeline([('s', StandardScaler()), ('svc', SVC(kernel='rbf'))])
print(f'CV accuracy = {cross_val_score(pipe, X, y, cv=5).mean():.4f}')


---

## Exercise 2

**Task 2.** Now fit *unscaled* RBF SVM on the same data. What
happens to CV accuracy?

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
acc = cross_val_score(SVC(kernel='rbf'), X, y, cv=5).mean()
print(f'unscaled CV accuracy = {acc:.4f}')
print('Without scaling, body_mass_g (in grams, ~3000-6000) dominates the')
print('RBF distance and the model essentially ignores the bill features.')


---

## Exercise 3

**Task 3.** Tune `C ∈ [0.1, 1, 10]` and `gamma ∈ ['scale', 0.1, 1.0]`
with a 5-fold `GridSearchCV` on the scaled pipeline.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(Pipeline([('s', StandardScaler()), ('svc', SVC(kernel='rbf'))]),
                     {'svc__C': [0.1, 1, 10], 'svc__gamma': ['scale', 0.1, 1.0]},
                     cv=5, n_jobs=-1).fit(X, y)
print(f'best params = {grid.best_params_}')
print(f'best CV acc = {grid.best_score_:.4f}')
